# Lección 9.5 — Optimizadores: SGD, Momentum, RMSProp y Adam, o cómo bajar la montaña con estilo

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.3 (descenso de gradiente), Lección 3.4 (tasa de aprendizaje), Lección 3.5 (mini-batch), Lección 9.4 (retropropagación y el bucle de entrenamiento)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con analogías qué añade cada optimizador al descenso de gradiente: **inercia** (Momentum), **pasos adaptados a cada perilla** (RMSProp) y **las dos cosas** (Adam).
- Ver, en un mapa 2D, por qué el descenso de gradiente simple **zigzaguea** en un valle alargado.
- Usar `torch.optim` y comparar optimizadores con curvas de pérdida.
- Elegir un optimizador y un valor de partida para su tasa de aprendizaje sin adivinar.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torchvision import datasets

np.random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: del excursionista a la bola de boliche

En la Lección 3.3 bajábamos la montaña como un **excursionista en la niebla**: mira la pendiente bajo sus pies y da un paso cuesta abajo. Ese es el **descenso de gradiente** (en redes se le llama **SGD**, por *Stochastic Gradient Descent*, porque usa mini-lotes, Lección 3.5).

Funciona, pero tiene un problema en los terrenos reales. Imagina un **cañón largo y estrecho**:

```
   paredes empinadas ↓↓            el fondo baja MUY suave hacia la salida →
  \    /\    /\    /\    /
   \  /  \  /  \  /  \  /   ......................................  🏁 mínimo
    \/    \/    \/    \/
   el excursionista rebota de pared en pared y avanza poquito
```

La pendiente más fuerte apunta **hacia la pared de enfrente**, no hacia la salida. El excursionista zigzaguea. Los optimizadores modernos le dan dos superpoderes:

| Superpoder | Analogía | Optimizador |
|---|---|---|
| **Inercia** | Una bola de boliche: acumula velocidad en la dirección que se repite y los rebotes laterales se cancelan | **Momentum** |
| **Pasos a la medida** | Zapatos inteligentes: pasos cortos donde el terreno es muy empinado, largos donde es plano | **RMSProp** |
| **Las dos cosas** | Bola de boliche con zapatos inteligentes 😄 | **Adam** |

### ✋ Ejercicio 1 — Calentamiento
Vas en bicicleta por un camino lleno de baches pequeños que te empujan a izquierda y derecha, pero el camino en general va **recto hacia el norte**. ¿Por qué ir rápido (con inercia) te ayuda a mantener el rumbo, mientras que ir muy despacio te hace tambalear?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con inercia, cada bache te desvía **un poquito**, pero tu velocidad acumulada hacia el norte domina: los empujones a izquierda y derecha **se cancelan** entre sí. Despacio, cada bache cambia tu dirección por completo. Momentum hace exactamente eso con los gradientes: promedia los pasos recientes, así que lo que se repite (ir hacia el mínimo) se refuerza y lo que cambia de signo (el zigzag) se cancela.
</details>

---
## 2. 📖 El terreno de pruebas: un valle alargado

Para **ver** a cada optimizador, usaremos una "pérdida" de solo **dos perillas**, `w1` y `w2`, con forma de cañón: muy empinada en la dirección de `w2` y casi plana en la de `w1`. El mínimo está en (0, 0) y salimos desde (−9, 2).

> 💡 En una red real hay millones de perillas y no podemos dibujar el terreno, pero los valles alargados son **muy comunes**: algunas perillas afectan mucho a la pérdida y otras muy poco.

In [ ]:
A, B = 0.05, 4.0                     # curvatura suave en w1, fuerte en w2

def perdida_valle(w):
    return 0.5 * (A * w[0] ** 2 + B * w[1] ** 2)

def gradiente_valle(w):              # "pendiente bajo los pies" en cada dirección
    return np.array([A * w[0], B * w[1]])

INICIO = np.array([-9.0, 2.0])

def recorrer(paso, pasos=50, **hp):
    """Aplica una regla de actualización `paso` varias veces y devuelve el camino recorrido."""
    w, estado, camino = INICIO.copy(), {}, [INICIO.copy()]
    for t in range(1, pasos + 1):
        w = paso(w, gradiente_valle(w), estado, t, **hp)
        camino.append(w.copy())
    return np.array(camino)

def dibujar(caminos):
    g1, g2 = np.meshgrid(np.linspace(-10, 3, 200), np.linspace(-3, 3, 200))
    plt.contour(g1, g2, 0.5 * (A * g1 ** 2 + B * g2 ** 2), levels=20, cmap="Greys", alpha=.5)
    for nombre, c in caminos.items():
        plt.plot(c[:, 0], c[:, 1], "o-", ms=3, label=f"{nombre} · pérdida final {perdida_valle(c[-1]):.3f}")
    plt.plot(0, 0, "r*", ms=15); plt.xlabel("w1 (dirección suave)"); plt.ylabel("w2 (dirección empinada)")
    plt.ylim(-3, 3); plt.legend(fontsize=8); plt.show()

### 📖 SGD: el excursionista
La regla de siempre (Lección 3.3): `w ← w − tasa × gradiente`.

In [ ]:
def sgd(w, g, estado, t, lr):
    return w - lr * g

dibujar({"SGD lr=0.45": recorrer(sgd, lr=0.45)})

### 🔮 Ejercicio 2 — El dilema de la tasa
Tras 50 pasos, SGD todavía está lejos del mínimo (la ⭐). Parece que la solución es subir la tasa de aprendizaje. Predice qué pasará con `lr=0.52` y con `lr=0.1`.

✍️ *Mis predicciones:*

In [ ]:
dibujar({"SGD lr=0.1": recorrer(sgd, lr=0.1),
         "SGD lr=0.45": recorrer(sgd, lr=0.45)})
camino = recorrer(sgd, lr=0.52)       # ✏️ prueba 0.5, 0.52, 0.55
print(f"SGD lr=0.52 → posición final w1={camino[-1, 0]:.2f}, w2={camino[-1, 1]:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **`lr=0.1`:** ya no rebota, pero avanza todavía más lento: tras 50 pasos sigue en w1 ≈ −7.
- **`lr=0.52`:** ¡**explota**! `w2` termina cerca de 94: en la dirección empinada cada paso se pasa del fondo y cae más alto en la pared contraria.

Ese es el dilema de SGD: la tasa la limita la dirección **más empinada** (si no, explota), pero con esa tasa la dirección **suave** avanza a paso de tortuga. **Una sola tasa no sirve para todas las perillas.** Los optimizadores siguientes atacan este problema.
</details>

---
## 3. 📖 Momentum: la bola de boliche

En lugar de dar un paso según la pendiente de **ahora**, Momentum mantiene una **velocidad** que acumula las pendientes recientes:

```
velocidad ← β × velocidad + gradiente      (β ≈ 0.9: "recuerda el 90 % de la velocidad anterior")
w         ← w − tasa × velocidad
```

- En la dirección suave, todas las pendientes apuntan hacia el mismo lado → la velocidad **crece**.
- En la dirección empinada, la pendiente cambia de signo en cada rebote → se **cancela**.

In [ ]:
def momentum(w, g, estado, t, lr, beta=0.9):
    estado["v"] = beta * estado.get("v", 0) + g
    return w - lr * estado["v"]

dibujar({"SGD lr=0.45": recorrer(sgd, lr=0.45),
         "Momentum lr=0.1, β=0.9": recorrer(momentum, lr=0.1)})

### ✏️ Ejercicio 3 — La perilla β
β decide **cuánta memoria** tiene la bola. Antes de ejecutar, predice:
- Con `β = 0`, ¿a qué optimizador se parece?
- Con `β = 0.99`, ¿qué crees que hará la bola cerca del mínimo?

✍️ *Mis predicciones:*

In [ ]:
dibujar({f"Momentum β={b}": recorrer(momentum, lr=0.1, beta=b) for b in [0, 0.5, 0.9, 0.99]})   # ✏️

<details><summary><b>👀 Ver respuesta</b></summary>

- **`β = 0`:** la velocidad es solo el gradiente actual → es **SGD** puro con `lr=0.1`: tras 50 pasos sigue en w1 ≈ −7 (pérdida ≈ 1.2).
- **`β = 0.5`:** algo más rápido (w1 ≈ −5.5, pérdida ≈ 0.75), pero con poca memoria sigue lejos.
- **`β = 0.9`:** llega muy cerca del mínimo en 50 pasos (pérdida ≈ 0.003). ✅
- **`β = 0.99`:** demasiada inercia: la bola **se pasa de largo** hasta w1 ≈ +7, al otro lado del mínimo, y termina con una pérdida ≈ 1.9, peor que con β = 0.5. Tarda muchísimo en frenar.

Por eso el valor de partida casi universal es **β = 0.9**.
</details>

---
## 4. 📖 RMSProp: zapatos inteligentes

RMSProp ataca el dilema por otro lado: le da a **cada perilla su propio tamaño de paso**. Lleva un promedio de qué tan grandes han sido los gradientes de cada perilla, y **divide** el paso entre ese tamaño:

```
s ← ρ × s + (1 − ρ) × gradiente²          (tamaño típico de la pendiente, por perilla)
w ← w − tasa × gradiente / √s
```

- Perilla con pendientes grandes (`w2`, empinada) → divide entre algo grande → **paso corto**.
- Perilla con pendientes chicas (`w1`, suave) → divide entre algo chico → **paso largo**.

Es como si el terreno se "aplanara": todas las direcciones avanzan a un ritmo parecido.

In [ ]:
def rmsprop(w, g, estado, t, lr, rho=0.9, eps=1e-8):
    estado["s"] = rho * estado.get("s", 0) + (1 - rho) * g ** 2
    return w - lr * g / (np.sqrt(estado["s"]) + eps)

dibujar({"SGD lr=0.45": recorrer(sgd, lr=0.45),
         "RMSProp lr=0.3": recorrer(rmsprop, lr=0.3)})

### 🔮 Ejercicio 4
Fíjate en la **forma** del camino de RMSProp. ¿Se parece al zigzag de SGD o va más directo? ¿En qué dirección se mueve al principio?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Sin zigzag.** El primer paso es en **diagonal**: avanza lo mismo (≈1) en `w1` que en `w2`, como si las dos perillas tuvieran la misma pendiente. En pocos pasos baja al fondo del cañón y después recorre el fondo **en línea recta** hacia el mínimo, con pasos largos en la dirección suave. Eso es justo lo que hace dividir entre el tamaño típico del gradiente: cada perilla da pasos de tamaño parecido, sin importar si su pendiente es grande o chica. Llega prácticamente al mínimo (pérdida ≈ 0.000).

> 📝 `eps` (épsilon) es un número diminuto que evita dividir entre cero. Casi nunca se toca.
</details>

---
## 5. 📖 Adam: inercia + pasos a la medida

**Adam** (*Adaptive Moment Estimation*, 2014) combina las dos ideas: la **velocidad** de Momentum (promedio de gradientes, con `β1 = 0.9`) y el **tamaño típico** de RMSProp (promedio de gradientes², con `β2 = 0.999`). Además corrige el arranque, cuando esos promedios todavía están "fríos".

Es el optimizador **por defecto** en deep learning: funciona razonablemente bien en casi todo sin mucho ajuste.

In [ ]:
def adam(w, g, estado, t, lr, b1=0.9, b2=0.999, eps=1e-8):
    estado["m"] = b1 * estado.get("m", 0) + (1 - b1) * g          # velocidad (Momentum)
    estado["v"] = b2 * estado.get("v", 0) + (1 - b2) * g ** 2     # tamaño típico (RMSProp)
    m_hat = estado["m"] / (1 - b1 ** t)                           # corrección del arranque
    v_hat = estado["v"] / (1 - b2 ** t)
    return w - lr * m_hat / (np.sqrt(v_hat) + eps)

caminos = {"SGD lr=0.45": recorrer(sgd, lr=0.45),
           "Momentum lr=0.1": recorrer(momentum, lr=0.1),
           "RMSProp lr=0.3": recorrer(rmsprop, lr=0.3),
           "Adam lr=0.5": recorrer(adam, lr=0.5)}
dibujar(caminos)

### ✋ Ejercicio 5 — El torneo en el valle
Mirando la gráfica y las pérdidas finales de la leyenda:
1. ¿Cuál llegó más lejos del mínimo?
2. Adam tiene inercia: ¿se nota en su camino? ¿Cómo?
3. En este valle concreto ganó RMSProp. ¿Significa eso que siempre es el mejor?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **SGD** (pérdida ≈ 0.21): sigue a medio camino.
2. Sí: su camino se **pasa un poco** del mínimo (llega hasta w1 ≈ +1.4) y vuelve con una curva suave, en lugar de clavarse en él. Esa "sobrecarrera" es la inercia de la bola. Aun así termina muy cerca (pérdida ≈ 0.009).
3. **No.** Este es un terreno de juguete, perfectamente liso y sin el ruido de los mini-lotes. En redes reales, el terreno es ruidoso y con millones de dimensiones; ahí la combinación de Adam suele ser más robusta. **Ningún optimizador gana siempre** (el "no hay almuerzo gratis" de la Lección 1.6, otra vez).
</details>

---
## 6. 📖 Ahora en una red de verdad: `torch.optim`

En PyTorch no hay que escribir nada de esto: cada optimizador es **una línea**, y el bucle de entrenamiento (Lección 9.4) no cambia.

```python
torch.optim.SGD(modelo.parameters(), lr=0.01)
torch.optim.SGD(modelo.parameters(), lr=0.01, momentum=0.9)
torch.optim.RMSprop(modelo.parameters(), lr=0.001)
torch.optim.Adam(modelo.parameters(), lr=0.001)
```

Vamos a entrenar la misma red de dígitos (imágenes 8×8) con cada uno, usando mini-lotes de 32 (Lección 3.5).

In [ ]:
digitos = load_digits()
X = torch.tensor(digitos.data / 16.0, dtype=torch.float32)
y = torch.tensor(digitos.target)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
criterio = nn.CrossEntropyLoss()

def entrenar(crear_opt, epocas=20, lote=32, X_tr=X_tr, y_tr=y_tr, X_te=X_te, y_te=y_te):
    """Entrena una red pequeña y devuelve (pérdida de entrenamiento por época, exactitud en prueba)."""
    torch.manual_seed(0)
    red = nn.Sequential(nn.Linear(X_tr.shape[1], 64), nn.ReLU(), nn.Linear(64, 10))
    opt = crear_opt(red.parameters())
    historia = []
    for epoca in range(epocas):
        orden, total = torch.randperm(len(X_tr)), 0
        for i in range(0, len(X_tr), lote):
            idx = orden[i:i + lote]
            opt.zero_grad()
            perdida = criterio(red(X_tr[idx]), y_tr[idx])
            perdida.backward()
            opt.step()
            total += perdida.item() * len(idx)
        historia.append(total / len(X_tr))
    with torch.no_grad():
        exactitud = (red(X_te).argmax(1) == y_te).float().mean().item()
    return historia, exactitud

### 🔮 Ejercicio 6 — ¿Quién baja primero?
Vamos a comparar 4 optimizadores con sus valores de partida típicos. Antes de ejecutar: ¿cuál crees que tendrá la pérdida **más alta** tras 20 épocas? ¿Y la más baja?

✍️ *Mis predicciones:*

In [ ]:
optimizadores = {
    "SGD (lr=0.01)":               lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD+Momentum (lr=0.01, 0.9)": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "RMSProp (lr=0.001)":          lambda p: torch.optim.RMSprop(p, lr=0.001),
    "Adam (lr=0.001)":             lambda p: torch.optim.Adam(p, lr=0.001),
}
for nombre, crear in optimizadores.items():
    historia, exactitud = entrenar(crear)
    plt.plot(historia, label=f"{nombre} · prueba {exactitud:.1%}")
    print(f"{nombre:<28} pérdida final {historia[-1]:.3f} · exactitud prueba {exactitud:.1%}")
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida de entrenamiento (log)"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **La más alta, de lejos: SGD sin momentum** (pérdida ≈ 1.4, prueba ≈ 82 %). Con `lr=0.01` y sin inercia, en 20 épocas apenas está a medio camino.
- Los otros tres terminan **muy parecidos**: pérdida ≈ 0.12–0.15 y ≈ 96 % en prueba. En pérdida, SGD+Momentum y RMSProp quedan apenas por delante de Adam en este caso; en prueba, prácticamente empatados.

Lección: añadir **inercia o pasos adaptativos** es lo que marca la gran diferencia. Entre los optimizadores "modernos" las diferencias son pequeñas, y cambian según el problema y la tasa elegida.
</details>

---
## 7. 📖 ¿Qué pasa si…? Adam no es magia: la tasa sigue mandando

Es tentador pensar: *"Adam ajusta los pasos solo, así que la tasa de aprendizaje da igual"*. **Falso.** Adam adapta el tamaño **relativo** entre perillas, pero la tasa sigue fijando la escala de **todos** los pasos.

### 🔮 Ejercicio 7
Entrenamos con Adam y cuatro tasas: `0.0001`, `0.001`, `0.01` y `0.1`. ¿Cuál crees que quedará peor? ¿Habrá alguna que **ni siquiera** aprenda bien?

✍️ *Mis predicciones:*

In [ ]:
for lr in [0.0001, 0.001, 0.01, 0.1]:          # ✏️ prueba 1.0
    historia, exactitud = entrenar(lambda p: torch.optim.Adam(p, lr=lr))
    plt.plot(historia, label=f"Adam lr={lr} · prueba {exactitud:.1%}")
    print(f"lr={lr:<7} pérdida final {historia[-1]:.3f} · exactitud prueba {exactitud:.1%}")
plt.yscale("log"); plt.xlabel("época"); plt.ylabel("pérdida de entrenamiento (log)"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **`lr=0.0001`:** demasiado tímida. Tras 20 épocas la pérdida sigue en ≈1.5 y la prueba en ≈85 %: no le dio tiempo.
- **`lr=0.001`** (el valor de partida): bien, ≈96 %.
- **`lr=0.01`:** ¡la mejor aquí! Pérdida ≈ 0.01 y ≈98 % en prueba. Para esta red pequeña, el valor de partida era conservador.
- **`lr=0.1`:** aprende (≈95 %), pero peor que con 0.01 y con una curva **a saltos** (la pérdida baja a ≈0.07 y luego rebota a ≈0.4): los pasos son demasiado grandes para asentarse en el fondo del valle.

Dos lecciones: (1) la tasa cambia el resultado incluso con Adam, y es la primera perilla a ajustar; (2) `1e-3` es un **punto de partida**, no el óptimo. Un barrido en potencias de 10 (Lección 9.6) encuentra el mejor valor en pocos intentos. ✏️ Con `lr=1.0` el entrenamiento se rompe por completo: la pérdida se queda clavada en ≈2.4 y la exactitud en ≈10 %, lo mismo que adivinar al azar.
</details>

---
## 8. 📖 Hiperparámetros y perillas

| Hiperparámetro | Optimizador | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|---|
| `lr` (tasa de aprendizaje) | Todos | Tamaño de los pasos | Más rápido… hasta oscilar o explotar (NaN) | Más estable pero lento; puede quedarse corto | Adam: `1e-3` · SGD+momentum: `0.01`–`0.1` |
| `momentum` (β) | SGD | Memoria de la velocidad | Más inercia: atraviesa mesetas, pero se pasa de largo | Menos inercia: más zigzag (0 = SGD puro) | `0.9` |
| `nesterov` | SGD | Momentum que "mira adelante" antes de dar el paso | — | — | `True` suele ayudar un poco con momentum |
| `betas` (β1, β2) | Adam / AdamW | Memoria de la velocidad (β1) y del tamaño típico (β2) | Promedios más suaves y lentos de reaccionar | Más reactivo y ruidoso | `(0.9, 0.999)` — casi nunca se tocan |
| `alpha` (ρ) | RMSprop | Memoria del tamaño típico | Más suave | Más reactivo | `0.99` en PyTorch |
| `eps` | Adam / RMSprop | Evita dividir entre cero | Pasos más parecidos a SGD | — | `1e-8` (no tocar) |
| `weight_decay` | Todos | Castigo a los pesos grandes (regularización L2, Lección 7.1) | Más regularización (riesgo de subajuste) | Menos regularización | `0` · con AdamW: `0.01` (Lección 9.8) |

### 🧠 Regla práctica
> **Empieza con Adam y `lr=1e-3`. Si necesitas exprimir el último punto (o el artículo que replicas usa SGD), prueba SGD con momentum 0.9 y busca su tasa.** De todas las perillas de esta tabla, la que más importa con diferencia es `lr` (Lección 9.6).

> 📝 **AdamW** es Adam con el *weight decay* aplicado "como debe ser" (separado del paso adaptativo). Hoy es el estándar para entrenar Transformers y LLMs (Módulos 12–13). Lo verás en la Lección 9.8.

### ✋ Ejercicio 8 — ¿Por qué tasas tan distintas?
El punto de partida de SGD (`0.01`–`0.1`) es 10–100 veces mayor que el de Adam (`0.001`). Con lo que viste en las secciones 4 y 5, ¿por qué crees que Adam necesita una tasa menor?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

En SGD el paso es `tasa × gradiente`, y los gradientes de una red suelen ser números **pequeños**, así que hace falta una tasa grande para avanzar. Adam **divide** el gradiente entre su tamaño típico, así que cada perilla da pasos de tamaño ≈ `tasa` sin importar qué tan chico sea su gradiente. Con `lr=0.1`, ¡cada peso se movería 0.1 en cada paso! Por eso Adam usa tasas más pequeñas.
</details>

---
## 9. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| La pérdida explota o aparece `NaN` | Tasa demasiado alta (muy común al copiar una tasa de SGD a Adam) | Divide `lr` entre 10 |
| La pérdida baja, pero lentísimo | Tasa demasiado baja, o SGD sin momentum | Sube `lr` ×10; añade `momentum=0.9`; prueba Adam |
| Cambiaste de optimizador y "ya no funciona" | Cada optimizador tiene su escala de tasa | Vuelve a buscar `lr` para el nuevo optimizador |
| Adam entrena genial pero valida peor que SGD | Adam tiende a generalizar un poco peor en algunos problemas de visión | Prueba AdamW con `weight_decay`, o SGD+momentum con un *scheduler* (Lección 9.6) |
| Crees que el optimizador "no hace nada" | Le pasaste los parámetros equivocados (de otro modelo, o antes de recrear el modelo) | `torch.optim.Adam(modelo.parameters(), ...)` **después** de crear el modelo definitivo |
| Al reanudar un entrenamiento guardado, la pérdida salta | Guardaste el modelo pero no el estado del optimizador (velocidades de Adam) | Guarda también `opt.state_dict()` |

### 🔮 Ejercicio 9
Un compañero pasa de `SGD(lr=0.1, momentum=0.9)`, que le funcionaba bien, a `Adam(lr=0.1)` "porque Adam es mejor". La pérdida se dispara. ¿Qué le dirías?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Que la tasa **no se traslada** de un optimizador a otro. `0.1` es razonable para SGD, pero para Adam es enorme (lo viste en el Ejercicio 7). Debe empezar con `Adam(lr=1e-3)` y, si quiere, buscar la mejor tasa con un barrido (Lección 9.6).
</details>

---
## 10. 📋 Ficha de los optimizadores

| | SGD | SGD + Momentum | RMSProp | Adam / AdamW |
|---|---|---|---|---|
| **Intuición** | Excursionista: paso según la pendiente de ahora | Bola de boliche con inercia | Zapatos inteligentes: paso a la medida de cada perilla | Bola con zapatos inteligentes |
| **Recuerda** | Nada | Velocidad (promedio de gradientes) | Tamaño típico (promedio de gradientes²) | Ambos |
| **Hiperparámetros clave** | `lr` | `lr`, `momentum` | `lr`, `alpha` | `lr`, `betas`, `weight_decay` |
| **Tasa de partida** | 0.01–0.1 | 0.01–0.1 (con 0.9) | 1e-3 | 1e-3 |
| **✅ Fortalezas** | Simple, poca memoria | Rápido en valles, muy usado en visión, generaliza bien | Robusto a escalas distintas | Funciona bien "de fábrica" casi siempre |
| **❌ Debilidades** | Lento, zigzaguea | Más sensible a la tasa que Adam | Hoy casi siempre lo supera Adam | Más memoria (2 promedios por peso); a veces generaliza algo peor |
| **Úsalo cuando…** | Para aprender o en problemas pequeños | Visión (CNN) con *scheduler*, cuando quieres exprimir el resultado | Redes recurrentes antiguas | **Por defecto**; Transformers y LLMs (AdamW) |

## 11. 🧠 Resumen en 5 puntos
1. Un **optimizador** decide cómo usar el gradiente (Lección 9.4) para mover los pesos; todos parten del descenso de gradiente.
2. SGD sufre en **valles alargados**: la dirección empinada limita la tasa y la dirección suave avanza lentísimo.
3. **Momentum** añade inercia (lo que se repite se refuerza, el zigzag se cancela); **RMSProp** da a cada perilla su propio tamaño de paso.
4. **Adam** combina las dos ideas y es el punto de partida por defecto (`lr=1e-3`); **AdamW** añade bien el *weight decay*.
5. Ningún optimizador elimina la necesidad de elegir bien la **tasa de aprendizaje**, y la tasa no se copia de un optimizador a otro.

---
## 12. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 12.1 Repaso rápido (opción múltiple)

**1.** En un valle alargado, SGD zigzaguea porque…
- a) la tasa de aprendizaje es demasiado baja
- b) el gradiente apunta sobre todo hacia la pared empinada, no hacia el mínimo
- c) el valle tiene muchos mínimos locales
- d) los datos no están normalizados

**2.** ¿Qué hace `momentum=0`?
- a) Detiene el entrenamiento · b) Lo convierte en SGD puro · c) Lo convierte en Adam · d) Duplica la tasa

**3.** ¿Qué idea añade RMSProp?
- a) Inercia
- b) Un tamaño de paso propio para cada perilla según el tamaño típico de su gradiente
- c) Apagar neuronas al azar
- d) Bajar la tasa con el tiempo

**4.** Tu red con Adam y `lr=0.1` da `NaN` en la segunda época. ¿Qué pruebas primero?
- a) Cambiar a RMSProp · b) Bajar `lr` a 0.001 · c) Subir `betas` · d) Entrenar más épocas

**5.** ¿Cuál es el punto de partida más razonable para un problema nuevo?
- a) SGD con `lr=1` · b) Adam con `lr=1e-3` · c) RMSProp con `lr=0.1` · d) Adam con `lr=1e-6`

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** La pendiente más fuerte apunta hacia la pared de enfrente, así que el paso rebota de un lado a otro.
2. **b)** Sin memoria de la velocidad, el paso es solo el gradiente actual.
3. **b)** Divide el paso entre el tamaño típico del gradiente de cada perilla.
4. **b)** Casi siempre un `NaN` temprano es una tasa demasiado alta. Divide entre 10 (o 100) antes de tocar nada más.
5. **b)** Funciona razonablemente bien en casi todo. Después ajustas.
</details>

### 12.2 Preguntas para pensar (sin ejecutar)

**A.** Adam guarda **dos** promedios por cada peso, además del peso mismo. Para un LLM de 7 mil millones de parámetros, ¿cuántos números tiene que guardar Adam en memoria aparte de los pesos? ¿Por qué crees que existen versiones de optimizadores que "ahorran memoria"?

**B.** Momentum ayuda a atravesar **mesetas** (zonas casi planas, Lección 3.6). Explica por qué con la analogía de la bola.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** 2 × 7 mil millones = **14 mil millones** de números extra: el doble de lo que ocupan los propios pesos. En memoria de GPU eso es muchísimo, por eso existen variantes de 8 bits o que guardan menos información, y técnicas como LoRA (Lección 13.7) que entrenan solo una fracción de los pesos.

**B.** En una meseta la pendiente es casi cero: el excursionista (SGD) da pasos diminutos y se queda casi quieto. La bola, en cambio, **llega con velocidad** acumulada de la bajada anterior y sigue rodando aunque el suelo sea plano, hasta encontrar otra bajada.
</details>

### 12.3 🧩 Práctica integradora: elige el optimizador para MNIST

Vas a elegir optimizador y tasa para reconocer dígitos escritos a mano de **MNIST** (imágenes de 28×28 = 784 píxeles), con un presupuesto fijo: **3 épocas**. Usaremos 6,000 imágenes para entrenar y 2,000 para validar. Hazlo en 5 pasos cortos y **no avances sin revisar cada respuesta**.

**Paso 1 ▶️ — Carga los datos**

In [ ]:
mnist = datasets.MNIST("data", train=True, download=True)
imagenes = mnist.data[:8000].float().div(255).view(-1, 784)   # píxeles 0..1, aplanados
etiquetas = mnist.targets[:8000]
Xm_tr, ym_tr = imagenes[:6000], etiquetas[:6000]
Xm_va, ym_va = imagenes[6000:], etiquetas[6000:]

def probar(crear_opt, epocas=3):
    historia, exactitud = entrenar(crear_opt, epocas=epocas, lote=64,
                                   X_tr=Xm_tr, y_tr=ym_tr, X_te=Xm_va, y_te=ym_va)
    return historia[-1], exactitud

print(f"Entrenamiento: {len(Xm_tr)} · validación: {len(Xm_va)} · píxeles por imagen: {Xm_tr.shape[1]}")

**Paso 2 🔮 — La línea base: SGD sin momentum.** Con `lr=0.01` y solo 3 épocas, ¿qué exactitud de validación esperas: ≈50 %, ≈80 % o ≈95 %?

✍️ *Mi apuesta:*

In [ ]:
perdida, exactitud = probar(lambda p: torch.optim.SGD(p, lr=0.01))
print(f"SGD lr=0.01 → pérdida {perdida:.3f} · exactitud validación {exactitud:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**74 %**, con la pérdida todavía en ≈1.6. No es que SGD no sirva: es que con esta tasa y sin inercia, 3 épocas no alcanzan. Es nuestra línea base.
</details>

**Paso 3 ✏️ — Añade momentum.** Cambia `MOMENTUM` (prueba 0.5 y 0.9) y ejecuta.

In [ ]:
MOMENTUM = 0.9   # ✏️
perdida, exactitud = probar(lambda p: torch.optim.SGD(p, lr=0.01, momentum=MOMENTUM))
print(f"SGD lr=0.01 momentum={MOMENTUM} → pérdida {perdida:.3f} · exactitud validación {exactitud:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `momentum=0.5` → ≈81 %.
- `momentum=0.9` → ≈89 % (pérdida ≈ 0.37).

Con la **misma** tasa y las mismas 3 épocas, solo añadir inercia sube ≈15 puntos. Un cambio de una palabra en el código.
</details>

**Paso 4 ▶️ — Un mini-torneo de tasas para Adam y SGD+momentum.** Probamos tres tasas para cada uno.

In [ ]:
for nombre, crear in [("Adam", lambda p, lr: torch.optim.Adam(p, lr=lr)),
                      ("SGD+mom", lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9))]:
    for lr in [0.0001, 0.001, 0.01, 0.1]:
        perdida, exactitud = probar(lambda p: crear(p, lr))
        print(f"{nombre:<8} lr={lr:<7} → pérdida {perdida:.3f} · validación {exactitud:.1%}")

✋ ¿Cuál es la mejor tasa para cada optimizador? ¿Coinciden?

<details><summary><b>👀 Ver respuesta</b></summary>

- **Adam:** mejor con `lr=0.01` (≈92 %). Con `0.1` se rompe (≈64 %) y con `0.0001` se queda corto (≈80 %).
- **SGD+momentum:** mejor con `lr=0.1` (≈92 %). Con `0.0001` casi no aprende nada (≈19 %, apenas mejor que adivinar al azar entre 10 dígitos).

**No coinciden:** la mejor tasa de SGD es 10 veces mayor que la de Adam, justo lo que vimos en el Ejercicio 8. Y bien ajustados, **los dos quedan prácticamente empatados**. Elegir bien la tasa importa más que elegir el optimizador.
</details>

**Paso 5 ✋ — Tu recomendación.** Con un presupuesto de 3 épocas, ¿qué optimizador y qué tasa usarías? Escribe tu recomendación en dos frases para un compañero, incluyendo **qué harías** si el problema cambiara (más datos, otra red).

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"Usaría Adam con `lr=0.01`, que dio ≈92 % en 3 épocas; SGD con momentum 0.9 y `lr=0.1` quedó casi igual, así que cualquiera sirve si la tasa está bien elegida. Si cambiara el problema, no copiaría la tasa: volvería a probar 3–4 tasas en potencias de 10 empezando por el valor de partida de cada optimizador."*

Lo importante es que tu recomendación esté **respaldada por los números** de validación y que reconozca que la tasa hay que volver a buscarla cuando el problema cambia (Lección 9.6).
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** Distill, *"Why Momentum Really Works"* (Gabriel Goh): mueve las perillas de momentum y mira la bola en vivo.
- 📘 **Lectura:** Sebastian Ruder, *"An overview of gradient descent optimization algorithms"*: el resumen clásico de todos los optimizadores.
- 📺 **Video:** Andrew Ng, *"Gradient Descent With Momentum"*, *"RMSprop"* y *"Adam Optimization Algorithm"* (DeepLearning.AI, inglés con subtítulos).
- 📘 **Documentación:** PyTorch, `torch.optim`.

---
⬅️ **Anterior:** [9.4 Retropropagación (*backpropagation*): repartir la culpa del error](9.4-retropropagacion.ipynb) · ➡️ **Siguiente:** [9.6 Hiperparámetros de la red: el tablero de perillas](9.6-hiperparametros-de-la-red.ipynb)